### load Dataset

In [2]:
import pandas as pd

df = pd.read_csv("patients.csv")

#### 🔍 2. **Start with Basic Exploration**

Use `.info()`, `.describe()`, `.head()`, `.tail()`, `.sample()`

```python
print(df.info())       # Shows column types and missing values
print(df.describe())   # Stats for numeric columns
print(df.head())       # Preview first few rows
```

**What to Look For**:

* `Non-null count` < total rows → **Missing Values**
* Weird data types (e.g., object for numbers)
* Large difference between min and max → **Outliers**

### checking information about dataset missing data and incorrect data types

In [3]:
print(df.info())       # Shows column types and missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1995 entries, 0 to 1994
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Name           1995 non-null   object 
 1   Age            1995 non-null   object 
 2   BloodPressure  1782 non-null   float64
 3   Diagnosis      1995 non-null   object 
dtypes: float64(1), object(3)
memory usage: 62.5+ KB
None


Age shouldnt be an object type. It should an integer type. 
The fact that it's `object` means it likely contains **non-numeric values**, such as:

* Missing values stored as strings like `"unknown"`, `"N/A"`, or empty strings `""`
* Inconsistent formats, like `"30 years"` instead of `30`
* Words instead of numbers, like `"forty"`

 **How to Fix It**

Here’s a step-by-step approach in code:

In [4]:
# Check unique values to identify bad entries
print(df['Age'].unique())

['23' '49' '19' '74' '85' '58' '81' '84' '69' '77' '30' '72' '88' '18'
 '38' 'thirty' '89' '42' '87' '73' '50' '61' '32' '70' '59' '36' '51' '82'
 '78' '47' '66' '24' '45' '76' 'twenty' '46' '34' '75' '41' '35' '27' '65'
 '80' '20' '55' '21' '37' '43' '53' '57' '29' '25' '44' '64' '67' '48'
 '63' '62' '31' '68' '71' '54' '79' '39' '40' '60' '56' '28' '26' '83'
 '33' '52' '86' '22' 'fifty' 'sixty' 'forty']


As suspected, the `Age` column is a mix of **numeric strings** (e.g., `'23'`, `'49'`) and **word-based numbers** like:

* `'thirty'`
* `'twenty'`
* `'forty'`
* `'fifty'`
* `'sixty'`

These word-based entries are the reason the column is stored as `object` instead of a numeric type.

---

### ✅ Here's How to Clean and Convert `Age` Properly

You can map the word-based ages to their numeric equivalents and then convert the column to `float` or `int`.

In [5]:
# Mapping for word-based numbers
word_to_num = {
    'twenty': 20,
    'thirty': 30,
    'forty': 40,
    'fifty': 50,
    'sixty': 60
}

# Replace word-based entries with corresponding numeric values
df['Age'] = df['Age'].replace(word_to_num)

# Now convert the entire column to numeric (this will coerce any remaining bad values to NaN)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# Confirm conversion
print(df.dtypes)
print(df['Age'].unique())

Name              object
Age                int64
BloodPressure    float64
Diagnosis         object
dtype: object
[23 49 19 74 85 58 81 84 69 77 30 72 88 18 38 89 42 87 73 50 61 32 70 59
 36 51 82 78 47 66 24 45 76 20 46 34 75 41 35 27 65 80 55 21 37 43 53 57
 29 25 44 64 67 48 63 62 31 68 71 54 79 39 40 60 56 28 26 83 33 52 86 22]


In [6]:
print(df.describe())   # Stats for numeric columns



               Age  BloodPressure
count  1995.000000    1782.000000
mean     53.249123     135.888328
std      20.507322      30.173937
min      18.000000      30.000000
25%      36.000000     112.000000
50%      53.000000     135.000000
75%      71.000000     157.000000
max      89.000000     400.000000


Based on the summary statistics, the `BloodPressure` column **contains outliers**.

---

### 🩺 **What Is the Ideal Range for Blood Pressure?**

According to medical guidelines (e.g., WHO, American Heart Association), **normal adult blood pressure** is generally considered to be:

| Category             | Systolic (upper) | Diastolic (lower) |
| -------------------- | ---------------- | ----------------- |
| Normal               | **< 120**        | **< 80**          |
| Elevated             | **120–129**      | **< 80**          |
| Hypertension Stage 1 | **130–139**      | **80–89**         |
| Hypertension Stage 2 | **140+**         | **90+**           |
| Hypertensive Crisis  | **180+**         | **120+**          |

In practice, **a realistic/physiological range** for systolic blood pressure values is roughly:

> **🟢 Normal range: 90–180 mmHg**
> Values **below 80** or **above 200** are often suspect (either outliers or entry errors).

---

### 🚨 The Stats

| Stat     | Value                  |
| -------- | ---------------------- |
| **Mean** | 135.9                  |
| **Min**  | 30.0 ❌ extremely low   |
| **Max**  | 400.0 ❌ extremely high |

So we definitely have:

* **Unrealistically low** values: `< 60`
* **Unrealistically high** values: `> 200` or especially `> 250`

---

What we Can Do

In [8]:
# 1. Detect and Flag Outliers

# Using an acceptable range (say 60 to 200):

# Define acceptable range
valid_range = (60, 200)

# Flag outliers
outliers = df[(df['BloodPressure'] < valid_range[0]) | (df['BloodPressure'] > valid_range[1])]
print(outliers)



         Name  Age  BloodPressure Diagnosis
56    charlie   55          400.0   OBESITY
454     alice   68          300.0   obesity
515     diana   82          400.0  Diabetes
643     FRANK   83          300.0    Asthma
899    Hannah   42          400.0   Obesity
1006    alice   81           30.0   HEALTHY
1054    Alice   26          400.0  Diabetes
1114  Charlie   45          400.0   HEALTHY
1489    FRANK   62          300.0  DIABETES
1785   HANNAH   77           30.0   OBESITY


#2. Handle Outliers

* **Option 1:** Remove them

```python
df = df[(df['BloodPressure'] >= 60) & (df['BloodPressure'] <= 200)]
```

* **Option 2:** Impute with median or cap values

```python
# Cap extreme values
df['BloodPressure'] = df['BloodPressure'].clip(lower=60, upper=200)
```

In [11]:
# Cap extreme values
df['BloodPressure'] = df['BloodPressure'].clip(lower=60, upper=200)



In [12]:
# Define acceptable range
valid_range = (60, 200)

# Flag outliers
outliers = df[(df['BloodPressure'] < valid_range[0]) | (df['BloodPressure'] > valid_range[1])]
print(outliers)



Empty DataFrame
Columns: [Name, Age, BloodPressure, Diagnosis]
Index: []
